# UrbanIQ | PBL Fase 5
## Inteligência Analítica, Estatística e Tomada de Decisão

**Desafio 10 — Centro de Operações Urbanas (COU) da cidade Alfa**

| | |
|---|---|
| **Grupo** | DataGuy |
| **Integrante** | Thiago Fiel de Oliveira — RM 570088 |
| **Turma** | 1TSCO |
| **Base de dados** | `cidade_alfa_ocorrencias_urbanas.xlsx` |

---

### A pergunta que este notebook responde

> **A cidade Alfa investiu em sensores inteligentes. Esse investimento chega até o cidadão?**

Para uma ocorrência sair do mundo real e virar serviço prestado, ela percorre quatro elos.
A cadeia é tão forte quanto o elo mais fraco, então cada elo será testado separadamente:

```
   ELO 1              ELO 2             ELO 3            ELO 4
  DETECÇÃO    →     DESPACHO     →    EXECUÇÃO    →   PERCEPÇÃO
 o sensor vê     a central aciona    a equipe         o cidadão
 o problema        a equipe           resolve           avalia
```

### Estrutura

| Parte | Conteúdo |
|---|---|
| **1** | 1º Desafio — preparação e qualidade dos dados |
| **2** | 2º Desafio — análise estatística e investigação |
| **3** | 3º Desafio — visualização e recomendações |

---
# PARTE 1 — 1º Desafio: Preparação e Qualidade dos Dados

## 1.1 Ambiente de trabalho

Bibliotecas usadas nesta parte:

- **pandas** — manipulação do DataFrame
- **numpy** — operações numéricas e tratamento de nulos
- **matplotlib** — visualizações estáticas do diagnóstico
- **openpyxl** — leitura e escrita do arquivo Excel

No Google Colab, pandas, numpy e matplotlib já vêm instalados.

In [ ]:
# openpyxl lê o Excel. pingouin e sympy são as ferramentas que o material
# da disciplina adota (Caps 9, 10, 13 e 14) e não vêm no Colab.
!pip install -q openpyxl pingouin sympy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

# Exibição mais confortável no Colab
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

print("Bibliotecas carregadas.")
print("pandas", pd.__version__, "| numpy", np.__version__)

### Identificação das capturas

O número que o Colab mostra entre colchetes, como `[7]`, é o **contador de execução**.
Ele aumenta toda vez que qualquer célula roda, então reexecutar uma célula muda o número
dela. Isso torna o contador inútil como referência estável.

Para resolver, cada célula que gera uma evidência para o relatório imprime a própria
identidade no topo da saída. Assim o print sai autoexplicativo e a numeração não depende
da ordem em que você executou nada.

As figuras também são gravadas em arquivo com nome fixo, em resolução alta, o que dá
qualidade muito melhor no PDF do que uma captura de tela.

In [ ]:
from pathlib import Path

PASTA_SAIDA = Path("saidas_fase5")
PASTA_SAIDA.mkdir(exist_ok=True)


def marcar(numero, titulo):
    """Imprime o cabeçalho de identificação da captura."""
    print("=" * 66)
    print(f"  CAPTURA {numero:02d}  |  {titulo}")
    print("=" * 66)


def salvar_figura(numero, nome):
    """Grava a figura atual em PNG de alta resolução."""
    caminho = PASTA_SAIDA / f"{numero:02d}-{nome}.png"
    plt.savefig(caminho, dpi=150, bbox_inches="tight")


print(f"Figuras serão gravadas em: {PASTA_SAIDA.resolve()}")

## 1.2 Carga do arquivo Excel

O arquivo tem **duas abas**:

- `dados` — as ocorrências registradas pelo Centro de Operações
- `dicionario_dados` — a descrição de cada coluna

Faça o upload de `cidade_alfa_ocorrencias_urbanas.xlsx` pelo painel de arquivos do Colab
antes de executar a célula abaixo.

In [ ]:
ARQUIVO = "cidade_alfa_ocorrencias_urbanas.xlsx"

df = pd.read_excel(ARQUIVO, sheet_name="dados")
dicionario = pd.read_excel(ARQUIVO, sheet_name="dicionario_dados")

# Cópia do estado original: usada no fim para medir o efeito da limpeza
df_original = df.copy()

print(f"Arquivo carregado: {ARQUIVO}")
print(f"Aba 'dados' ............ {df.shape[0]:,} linhas x {df.shape[1]} colunas")
print(f"Aba 'dicionario_dados' . {dicionario.shape[0]} colunas documentadas")

### Dicionário de dados

Antes de tocar em qualquer número, é preciso entender o que cada coluna significa.
Analisar uma base sem dicionário é o erro mais comum de quem está começando.

In [ ]:
marcar(1, "Dicionário de dados da base")
dicionario

## 1.3 Conhecendo a base

Quatro comandos respondem às perguntas iniciais de qualquer cientista de dados:

| Comando | Pergunta que responde |
|---|---|
| `df.shape` | Quantos registros e quantas colunas eu tenho? |
| `df.columns` | Quais são os atributos disponíveis? |
| `df.info()` | Qual o tipo de cada coluna e quantos valores estão preenchidos? |
| `df.head()` | Como os dados se parecem na prática? |

In [ ]:
# Quantidade de linhas e colunas disponíveis
df.shape

In [ ]:
# Nomes das colunas disponíveis no dataframe
df.columns

In [ ]:
# Tipos de dados e quantidade de valores preenchidos por coluna
marcar(2, "Estrutura da base: tipos e preenchimento")
df.info()

In [ ]:
# Primeiros registros da base
df.head(10)

### Primeira leitura

A base reúne **cinco fontes de dados diferentes** em uma única tabela, que é exatamente o que
o Desafio 10 pede ao falar em "unificar diversas fontes":

- `APP_MOBILE`, `PORTAL_WEB`, `TELEFONE_156` — registros feitos por cidadãos
- `SENSOR_IOT`, `CAMERA_HD` — detecções automáticas

A coluna `tipo_fonte` agrupa essas cinco origens em duas naturezas, **CIDADAO** e **AUTOMATICA**.
É essa separação que vai permitir responder à pergunta central do notebook.

In [ ]:
# Como as ocorrências se distribuem entre as fontes de dados
print("Por fonte de dado:")
print(df["fonte_dado"].value_counts())
print()
print("Por natureza da fonte:")
print(df["tipo_fonte"].value_counts())

---
## 1.4 Diagnóstico de qualidade

Agora o foco muda: sai o "o que eu tenho" e entra o **"em que posso confiar"**.

Dados de sensores e de registros manuais quase nunca chegam limpos. Falhas de integração,
relógios dessincronizados, campos não preenchidos e reenvios de lote são rotina em qualquer
Centro de Operações real.

Vamos procurar seis tipos de problema:

1. Valores nulos
2. Registros duplicados
3. Valores fora de faixa válida
4. Inconsistências de data
5. Falta de padronização em texto
6. Outliers extremos

### Problema 1 — Valores nulos

In [ ]:
# Quantidade e percentual de valores nulos por coluna
marcar(3, "Valores nulos por coluna")

nulos = pd.DataFrame({
    "qtd_nulos": df.isnull().sum(),
    "pct_nulos": (df.isnull().sum() / len(df) * 100).round(2)
})
nulos[nulos["qtd_nulos"] > 0].sort_values("qtd_nulos", ascending=False)

**Reflexão de negócio.** Nem todo nulo é um defeito, e essa distinção é o ponto mais
importante desta etapa.

| Coluna | Nulo significa | É defeito? |
|---|---|---|
| `dt_encerramento` | A ocorrência ainda está em aberto | **Não.** É informação legítima |
| `tempo_resolucao_horas` | Consequência do item acima | **Não** |
| `satisfacao_cidadao` | Ocorrência não finalizada ou cidadão não respondeu | **Parcial** |
| `bairro` | O endereço não foi confirmado no atendimento | **Sim** |

Tratar todo nulo da mesma forma, apagando linhas, destruiria justamente as ocorrências
em aberto, que são as mais urgentes para um Centro de Operações. Vamos confirmar essa
hipótese cruzando os nulos com o status.

In [ ]:
# Os nulos de dt_encerramento correspondem mesmo às ocorrências não finalizadas?
marcar(4, "Nulo legítimo x nulo defeituoso: status x dt_encerramento")

pd.crosstab(
    df["status_ocorrencia"],
    df["dt_encerramento"].isnull(),
    colnames=["dt_encerramento é nulo"]
)

Confirmado: os nulos de `dt_encerramento` aparecem **somente** nos status
`ABERTO`, `EM_ANALISE` e `EM_ATENDIMENTO`. Não é sujeira, é o ciclo de vida da ocorrência.

### Problema 2 — Registros duplicados

In [ ]:
# Registros exatamente iguais em todas as colunas
qtd_duplicados = df.duplicated().sum()
print(f"Registros duplicados: {qtd_duplicados}")
print(f"Percentual da base: {qtd_duplicados / len(df) * 100:.2f}%")

In [ ]:
# Exemplo de um par duplicado, para entender o que aconteceu
exemplo = df[df.duplicated(keep=False)].sort_values("id_ocorrencia").head(4)
exemplo[["id_ocorrencia", "fonte_dado", "dt_abertura", "bairro", "categoria", "score_prioridade"]]

**Causa provável.** As duplicatas repetem inclusive o `id_ocorrencia`, que deveria ser único.
Isso não é o cidadão abrindo o mesmo chamado duas vezes, e sim um **reenvio de lote** na
integração entre o sistema de origem e o COU. Se não forem removidas, todas as contagens
e médias ficam infladas.

### Problema 3 — Valores fora de faixa válida

O `describe()` expõe mínimos e máximos impossíveis.

In [ ]:
# Estatísticas iniciais das colunas numéricas
df.describe()

Três anomalias saltam aos olhos:

- `score_prioridade` com **máximo 999**, quando a regra de negócio RN06 define a faixa de 0 a 100
- `tempo_resposta_min` com **valores negativos**, o que é fisicamente impossível
- `custo_operacional_reais` com máximo na casa dos milhões, muito acima da mediana

Vamos quantificar cada um.

In [ ]:
# Scores fora da faixa 0 a 100 definida pela RN06
fora_faixa = df[df["score_prioridade"] > 100]
print(f"Registros com score_prioridade > 100: {len(fora_faixa)}")
print("Valores encontrados:", sorted(fora_faixa["score_prioridade"].unique()))

In [ ]:
# Tempos de resposta negativos
negativos = df[df["tempo_resposta_min"] < 0]
print(f"Registros com tempo_resposta_min negativo: {len(negativos)}")
print(f"Faixa dos valores: de {negativos['tempo_resposta_min'].min()} a {negativos['tempo_resposta_min'].max()} min")

In [ ]:
# Avaliações fora da escala de 1 a 5
marcar(5, "Satisfação do cidadão: valor 9 em escala de 1 a 5")

escala = df["satisfacao_cidadao"].value_counts(dropna=False).sort_index()
print("Distribuição de satisfacao_cidadao:")
print(escala)

O valor **9** aparece em uma escala que vai de 1 a 5. É o clássico código sentinela que
a ferramenta de pesquisa usa para "não respondeu" e que ninguém traduziu na integração.
Se entrar na média, infla artificialmente a satisfação da cidade.

### Problema 4 — Inconsistências de data

Duas verificações lógicas: nenhuma ocorrência pode ser encerrada antes de ser aberta,
e nenhuma pode ser aberta no futuro.

In [ ]:
# Encerramento anterior à abertura
data_invertida = df[df["dt_encerramento"] < df["dt_abertura"]]
print(f"Registros com dt_encerramento anterior a dt_abertura: {len(data_invertida)}")

# Abertura em data futura (a base cobre até 30/09/2026)
LIMITE = pd.Timestamp("2026-09-30 23:59:59")
data_futura = df[df["dt_abertura"] > LIMITE]
print(f"Registros com dt_abertura no futuro: {len(data_futura)}")
if len(data_futura):
    print(f"Datas encontradas: {data_futura['dt_abertura'].dt.date.unique()}")

### Problema 5 — Falta de padronização em texto

Cada canal grava o nome da própria fonte de um jeito. Sem padronizar, o Python entende
`APP_MOBILE` e ` app_mobile ` como duas categorias diferentes, e qualquer agrupamento sai errado.

In [ ]:
# Todos os valores distintos encontrados na coluna fonte_dado
marcar(6, "Padronização: 10 rótulos distintos para 5 fontes reais")

print(f"Valores distintos em fonte_dado: {df['fonte_dado'].nunique()}")
print()
for valor in sorted(df["fonte_dado"].unique()):
    print(f"  [{valor}]  ->  {(df['fonte_dado'] == valor).sum()} registros")

São **10 rótulos** para apenas **5 fontes** reais. Os colchetes no `print` revelam os
espaços em branco no início e no fim, que passariam despercebidos numa leitura comum.

### Problema 6 — Outliers extremos

Nem todo outlier é erro. Um custo alto pode ser uma obra grande de verdade. O critério
aqui é a plausibilidade: valores milhares de vezes acima da mediana indicam erro de digitação.

In [ ]:
# Comparação entre mediana e valores extremos de custo
print(f"Mediana do custo operacional ... R$ {df['custo_operacional_reais'].median():,.2f}")
print(f"Percentil 99 .................. R$ {df['custo_operacional_reais'].quantile(0.99):,.2f}")
print(f"Máximo ........................ R$ {df['custo_operacional_reais'].max():,.2f}")
print()
extremos = df[df["custo_operacional_reais"] > 1_000_000]
print(f"Registros acima de R$ 1 milhão: {len(extremos)}")

In [ ]:
# Visualização do problema: com e sem os valores absurdos
marcar(7, "Efeito do outlier extremo na leitura do boxplot")

fig, eixos = plt.subplots(1, 2)

eixos[0].boxplot(df["custo_operacional_reais"].dropna(), vert=True)
eixos[0].set_title("Custo operacional (base bruta)")
eixos[0].set_ylabel("R$")

sem_absurdo = df[df["custo_operacional_reais"] < 1_000_000]["custo_operacional_reais"]
eixos[1].boxplot(sem_absurdo.dropna(), vert=True)
eixos[1].set_title("Custo operacional (sem os valores absurdos)")
eixos[1].set_ylabel("R$")

plt.tight_layout()
salvar_figura(7, "boxplot-custo")
plt.show()

O gráfico da esquerda é ilegível: três registros errados achatam toda a distribuição real
contra o eixo. É a demonstração visual de por que outlier extremo precisa ser tratado antes
de qualquer análise.

---
## 1.5 Limpeza e tratamento

O diagnóstico terminou. Agora cada problema recebe uma decisão, e **cada decisão precisa
de uma justificativa de negócio**, não apenas técnica.

Um princípio guia esta etapa:

> **Apague a célula, não a linha.**

Quando apenas um campo está inválido, remover a linha inteira joga fora dezenas de
informações válidas. Só removemos a linha quando o registro inteiro perde o sentido.

| # | Problema | Decisão | Justificativa |
|---|---|---|---|
| 1 | Duplicatas exatas | Remover a linha | Reenvio de lote. Manter infla todas as contagens |
| 2 | `fonte_dado` sem padrão | Padronizar o texto | São 5 fontes, não 10. Agrupamento depende disso |
| 3 | `bairro` nulo | Preencher com "Não Informado" | O resto do registro é válido e útil |
| 4 | `score_prioridade` = 999 | Converter para nulo | Valor sentinela. Não dá para inventar o score real |
| 5 | `satisfacao_cidadao` = 9 | Converter para nulo | Fora da escala 1 a 5. Entraria na média |
| 6 | `tempo_resposta_min` negativo | Converter para nulo | Relógio dessincronizado. O sinal não é recuperável com segurança |
| 7 | Encerramento antes da abertura | Anular as métricas de tempo | Sequência impossível. Os demais campos seguem válidos |
| 8 | Abertura em data futura | Remover a linha | Sem data confiável, o registro não se posiciona no tempo |
| 9 | Custo absurdo | Converter para nulo | Erro de digitação em ordem de serviço |

In [ ]:
# Trabalhamos sobre uma cópia, preservando df para comparação posterior
dfc = df.copy()
registro_limpeza = []   # guarda o efeito de cada etapa para o relatório final


def registrar(etapa, afetados, linhas_antes, linhas_depois):
    registro_limpeza.append({
        "etapa": etapa,
        "registros_afetados": afetados,
        "linhas_antes": linhas_antes,
        "linhas_depois": linhas_depois
    })
    print(f"{etapa}: {afetados} registro(s) tratado(s) | linhas: {linhas_antes:,} -> {linhas_depois:,}")

### Etapa 1 — Remover duplicatas exatas

In [ ]:
antes = len(dfc)
qtd = dfc.duplicated().sum()
dfc = dfc.drop_duplicates()
registrar("1. Duplicatas removidas", qtd, antes, len(dfc))

# Confirmação
print(f"Duplicatas restantes: {dfc.duplicated().sum()}")

### Etapa 2 — Padronizar a coluna `fonte_dado`

Três operações encadeadas: remover espaços das pontas, converter para maiúsculas e
trocar espaços internos por sublinhado.

In [ ]:
antes = len(dfc)
rotulos_antes = dfc["fonte_dado"].nunique()

dfc["fonte_dado"] = (
    dfc["fonte_dado"]
    .str.strip()
    .str.upper()
    .str.replace(" ", "_", regex=False)
)

registrar("2. fonte_dado padronizada", rotulos_antes - dfc["fonte_dado"].nunique(), antes, len(dfc))
print(f"Rótulos: {rotulos_antes} -> {dfc['fonte_dado'].nunique()}")
print(dfc["fonte_dado"].value_counts())

### Etapa 3 — Bairro não informado

Mesma decisão adotada no exercício guiado: o registro continua útil para análises de
tempo, categoria e fonte. Apagá-lo seria perder informação boa por causa de um campo.

In [ ]:
antes = len(dfc)
qtd = dfc["bairro"].isnull().sum()

dfc["bairro"] = dfc["bairro"].fillna("Não Informado")

registrar("3. bairro nulo preenchido", qtd, antes, len(dfc))
print(f"Nulos restantes em bairro: {dfc['bairro'].isnull().sum()}")

### Etapa 4 — Score de prioridade fora da faixa

In [ ]:
antes = len(dfc)
mascara = dfc["score_prioridade"] > 100
qtd = mascara.sum()

dfc.loc[mascara, "score_prioridade"] = np.nan

registrar("4. score_prioridade = 999 anulado", qtd, antes, len(dfc))
print(f"Faixa atual do score: {dfc['score_prioridade'].min():.0f} a {dfc['score_prioridade'].max():.0f}")

### Etapa 5 — Avaliação fora da escala

In [ ]:
antes = len(dfc)
mascara = ~dfc["satisfacao_cidadao"].isin([1, 2, 3, 4, 5]) & dfc["satisfacao_cidadao"].notna()
qtd = mascara.sum()

dfc.loc[mascara, "satisfacao_cidadao"] = np.nan

registrar("5. satisfacao fora da escala anulada", qtd, antes, len(dfc))
print("Valores válidos restantes:", sorted(dfc["satisfacao_cidadao"].dropna().unique()))

### Etapa 6 — Tempo de resposta negativo

Poderíamos aplicar o valor absoluto, supondo inversão de sinal. Mas isso é **suposição**,
não evidência: o relógio pode ter derivado alguns minutos ou algumas horas. Diante da
dúvida, anular é mais honesto que inventar.

In [ ]:
antes = len(dfc)
mascara = dfc["tempo_resposta_min"] < 0
qtd = mascara.sum()

dfc.loc[mascara, "tempo_resposta_min"] = np.nan

registrar("6. tempo_resposta negativo anulado", qtd, antes, len(dfc))
print(f"Mínimo atual: {dfc['tempo_resposta_min'].min():.0f} min")

### Etapa 7 — Encerramento anterior à abertura

Aqui só as métricas de tempo são anuladas. Categoria, bairro, fonte e score continuam
válidos e seguem servindo para as demais análises.

In [ ]:
antes = len(dfc)
mascara = dfc["dt_encerramento"] < dfc["dt_abertura"]
qtd = mascara.sum()

dfc.loc[mascara, ["dt_encerramento", "tempo_resolucao_horas"]] = np.nan
dfc.loc[mascara, "sla_cumprido"] = "NAO APURADO"

registrar("7. datas invertidas anuladas", qtd, antes, len(dfc))
print(f"Inconsistências restantes: {(dfc['dt_encerramento'] < dfc['dt_abertura']).sum()}")

### Etapa 8 — Abertura em data futura

Este é o único caso de remoção de linha. Sem data confiável de abertura, o registro não
se posiciona na linha do tempo, e toda a análise temporal, a derivada e a integral
dependem disso.

In [ ]:
antes = len(dfc)
LIMITE = pd.Timestamp("2026-09-30 23:59:59")
qtd = (dfc["dt_abertura"] > LIMITE).sum()

dfc = dfc[dfc["dt_abertura"] <= LIMITE]

registrar("8. registros com data futura removidos", qtd, antes, len(dfc))
print(f"Período coberto: {dfc['dt_abertura'].min().date()} a {dfc['dt_abertura'].max().date()}")

### Etapa 9 — Custo operacional absurdo

In [ ]:
antes = len(dfc)
mascara = dfc["custo_operacional_reais"] > 1_000_000
qtd = mascara.sum()

dfc.loc[mascara, "custo_operacional_reais"] = np.nan

registrar("9. custo absurdo anulado", qtd, antes, len(dfc))
print(f"Novo máximo: R$ {dfc['custo_operacional_reais'].max():,.2f}")
print(f"Nova mediana: R$ {dfc['custo_operacional_reais'].median():,.2f}")

---
## 1.6 Engenharia de atributos

A base guarda o **fato bruto**. Os indicadores que a análise precisa são **calculados** aqui,
e não gravados na planilha. Essa separação é uma boa prática: se a regra de cálculo mudar,
o dado de origem continua intacto.

Quatro atributos novos:

| Atributo | Cálculo | Para que serve |
|---|---|---|
| `lag_despacho_min` | abertura até despacho | Mede o **elo 2**, onde suspeitamos da quebra |
| `razao_sla` | tempo de resolução ÷ prazo prometido | Mede a promessa cumprida, não o tempo absoluto |
| `hora_do_dia` | hora da abertura | Identifica os picos de demanda |
| `dia_semana` | dia da semana da abertura | Separa rotina útil de fim de semana |

In [ ]:
# Elo 2: quanto tempo a ocorrência espera até a equipe ser acionada
dfc["lag_despacho_min"] = (
    (dfc["dt_despacho"] - dfc["dt_abertura"]).dt.total_seconds() / 60
).round(1)

# Elo 4: razão entre o tempo gasto e o prazo prometido
# Valor 1.0 = prazo cumprido no limite. Acima de 1.0 = promessa quebrada
dfc["razao_sla"] = (dfc["tempo_resolucao_horas"] / dfc["sla_horas_previsto"]).round(3)

# Sazonalidade intradiária e semanal
dfc["hora_do_dia"] = dfc["dt_abertura"].dt.hour
dias = {0: "1-Seg", 1: "2-Ter", 2: "3-Qua", 3: "4-Qui", 4: "5-Sex", 5: "6-Sab", 6: "7-Dom"}
dfc["dia_semana"] = dfc["dt_abertura"].dt.dayofweek.map(dias)
dfc["mes_ano"] = dfc["dt_abertura"].dt.to_period("M").astype(str)

print("Atributos criados:")
dfc[["lag_despacho_min", "razao_sla", "hora_do_dia", "dia_semana", "mes_ano"]].head()

In [ ]:
# Sanidade dos novos atributos
print("lag_despacho_min:")
print(f"  mediana {dfc['lag_despacho_min'].median():.1f} min | máximo {dfc['lag_despacho_min'].max():.0f} min")
print()
print("razao_sla (finalizadas):")
print(f"  mediana {dfc['razao_sla'].median():.2f}")
print(f"  dentro do prazo (<= 1.0): {(dfc['razao_sla'] <= 1).sum():,} ocorrências")
print(f"  fora do prazo  (> 1.0): {(dfc['razao_sla'] > 1).sum():,} ocorrências")

---
## 1.7 Relatório da limpeza

O que mudou entre a base bruta e a base tratada.

In [ ]:
marcar(8, "Relatório das 9 etapas de limpeza")

relatorio = pd.DataFrame(registro_limpeza)
relatorio

In [ ]:
# Comparação direta entre antes e depois
marcar(9, "Base bruta x base tratada")

comparacao = pd.DataFrame({
    "Base bruta": [
        len(df_original),
        df_original.duplicated().sum(),
        df_original["fonte_dado"].nunique(),
        (df_original["score_prioridade"] > 100).sum(),
        (df_original["tempo_resposta_min"] < 0).sum(),
        df_original["bairro"].isnull().sum(),
        f"R$ {df_original['custo_operacional_reais'].max():,.0f}",
    ],
    "Base tratada": [
        len(dfc),
        dfc.duplicated().sum(),
        dfc["fonte_dado"].nunique(),
        (dfc["score_prioridade"] > 100).sum(),
        (dfc["tempo_resposta_min"] < 0).sum(),
        dfc["bairro"].isnull().sum(),
        f"R$ {dfc['custo_operacional_reais'].max():,.0f}",
    ]
}, index=[
    "Total de registros",
    "Registros duplicados",
    "Rótulos distintos de fonte",
    "Scores fora da faixa 0-100",
    "Tempos de resposta negativos",
    "Bairros nulos",
    "Custo máximo",
])
comparacao

In [ ]:
perda = (1 - len(dfc) / len(df_original)) * 100
print(f"Base bruta ....... {len(df_original):,} registros")
print(f"Base tratada ..... {len(dfc):,} registros")
print(f"Perda de linhas .. {perda:.2f}%")
print()
print("A perda é baixa porque a maior parte dos problemas foi tratada no nível da célula,")
print("preservando as informações válidas de cada registro.")

### Conclusão da Parte 1

A base saiu de **10.180 registros com nove tipos de inconsistência** para uma base tratada,
documentada e confiável.

Três decisões merecem destaque na avaliação:

1. **Nulo legítimo foi separado de nulo defeituoso.** As ocorrências em aberto foram
   preservadas, e elas são justamente as mais relevantes para um Centro de Operações.
2. **A limpeza atuou na célula, não na linha.** Isso preservou informação válida que uma
   limpeza apressada teria descartado.
3. **Cada decisão foi justificada pelo negócio.** Nenhum registro foi removido apenas por
   ser incômodo para o código.

A base está pronta para a análise estatística da Parte 2, em que os quatro elos da cadeia
de valor do COU serão testados um a um.

In [ ]:
# Exporta a base tratada, insumo da Parte 2 e do dashboard
ARQUIVO_AJUSTADO = "cidade_alfa_ocorrencias_ajustado.xlsx"
dfc.to_excel(ARQUIVO_AJUSTADO, index=False)

print(f"Arquivo gerado: {ARQUIVO_AJUSTADO}")
print(f"{dfc.shape[0]:,} linhas x {dfc.shape[1]} colunas")

---
---

# PARTE 2 — 2º Desafio: Análise estatística e investigação dos dados

A base está limpa. Agora começa a investigação.

### A pergunta desta parte

> **A cidade Alfa investiu em sensores inteligentes. Esse investimento chega até o cidadão?**

Para uma ocorrência sair do mundo real e virar serviço prestado, ela percorre quatro
elos. A cadeia é tão forte quanto o elo mais fraco, então cada um será testado
separadamente:

```
   ELO 1              ELO 2             ELO 3            ELO 4
  DETECÇÃO    →     DESPACHO     →    EXECUÇÃO    →   PERCEPÇÃO
 o sensor vê     a central aciona    a equipe         o cidadão
 o problema        a equipe           resolve           avalia
```

| Seção | Elo | Técnica estatística |
|---|---|---|
| 2.1 | — | Estatística descritiva |
| 2.2 | — | Amostragem, dimensionamento e intervalo de confiança |
| 2.3 | 1 | Teste t de Student (scipy e pingouin) |
| 2.4 | 2 | Correlação e qui-quadrado |
| 2.5 | 3 | ANOVA (scipy e pingouin) e V de Cramér |
| 2.6 | 4 | Correlação de Pearson, Spearman e Kendall, e regressão |
| 2.7 | — | Limites, derivadas e integrais com SymPy |
| 2.8 | — | Armadilhas estatísticas |

As ferramentas seguem as adotadas na disciplina: `scipy.stats` e `pingouin` para
inferência (Caps 13 e 14), `sympy` para cálculo simbólico (Caps 9 e 10) e o
dimensionamento amostral do Cap 12.

## 2.0 Ferramentas adicionais

O DataFrame `dfc`, tratado na Parte 1, continua em memória. Aqui entram apenas as
bibliotecas de estatística inferencial e de cálculo simbólico, além da paleta de
cores usada nos gráficos.

In [ ]:
import sympy as smp
import scipy
import sklearn
from scipy import stats as st
from sklearn.model_selection import train_test_split
import pingouin as pg

# ---------------------------------------------------------------------------
# Identidade visual UrbanIQ (paleta validada para daltonismo e contraste)
# ---------------------------------------------------------------------------
OURO, ROXO, TERRACOTA, AZUL, OLIVA = "#A07400", "#7B3FA5", "#CC3B14", "#3160C8", "#5E8A00"
CATEGORICA = [OURO, ROXO, TERRACOTA, AZUL, OLIVA]
TINTA, SUAVE, PAPEL = "#2E2A26", "#6E655B", "#F6F3EC"

plt.rcParams.update({
    "figure.figsize": (11, 5),
    "figure.facecolor": PAPEL,
    "axes.facecolor": PAPEL,
    "axes.edgecolor": SUAVE,
    "axes.labelcolor": TINTA,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.color": SUAVE,
    "text.color": TINTA,
    "xtick.color": SUAVE,
    "ytick.color": SUAVE,
    "legend.frameon": False,
})

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")


def col_pg(tabela, *nomes):
    """Devolve a primeira coluna existente entre os nomes informados.

    O pingouin mudou os nomes das colunas entre versões: o que era 'cohen-d',
    'p-val' e 'p-unc' virou 'cohen_d', 'p_val' e 'p_unc'. Esta função deixa o
    notebook funcionar nas duas versões.
    """
    for nome in nomes:
        if nome in tabela.columns:
            return tabela[nome]
    raise KeyError(f"nenhuma destas colunas existe: {nomes}")


print("Ferramentas da Parte 2 carregadas. Versões em uso:")
print(f"  pandas {pd.__version__} | numpy {np.__version__} | scipy {scipy.__version__}")
print(f"  pingouin {pg.__version__} | sympy {smp.__version__} | scikit-learn {sklearn.__version__}")
print()
print(f"DataFrame herdado da Parte 1: {dfc.shape[0]:,} linhas x {dfc.shape[1]} colunas")

---
## 2.1 Estatística descritiva

Antes de testar qualquer hipótese é preciso conhecer o comportamento das variáveis.
A descritiva responde três perguntas: qual é o valor típico, quanta variação existe
em torno dele, e a distribuição é simétrica ou puxada para um lado.

Essa última pergunta é a mais importante e a mais ignorada. Ela decide se a média
é uma medida honesta ou enganosa.

In [ ]:
marcar(10, "Estatística descritiva das variáveis da cadeia")

variaveis = [
    "tempo_resposta_min", "lag_despacho_min", "tempo_resolucao_horas",
    "razao_sla", "score_prioridade", "satisfacao_cidadao",
]

descritiva = dfc[variaveis].describe().T
descritiva["assimetria"] = dfc[variaveis].skew()
descritiva["cv_%"] = (descritiva["std"] / descritiva["mean"] * 100).round(1)
descritiva.round(2)

**Como ler esta tabela.**

O **coeficiente de variação** (`cv_%`) é o desvio padrão em proporção da média. Acima
de 50% indica um processo instável: o tempo de atendimento não é confiável, varia
demais de um caso para outro. Para uma central de operações, isso é um problema por
si só, independente da média estar boa.

A **assimetria** positiva revela que a média está sendo puxada por uma cauda de casos
extremos. Quando ela aparece, a **mediana descreve melhor o caso típico** do que a
média. É por isso que vamos reportar as duas.

In [ ]:
marcar(11, "Distribuição do tempo de resolução e dispersão por região")

fig, eixos = plt.subplots(1, 2, figsize=(13, 4.8))

# Esquerda: histograma com média e mediana marcadas
dados = dfc["tempo_resolucao_horas"].dropna()
eixos[0].hist(dados, bins=60, color=OURO, edgecolor=PAPEL, linewidth=0.5)
eixos[0].axvline(dados.mean(), color=TERRACOTA, linewidth=2,
                 label=f"Média: {dados.mean():.1f} h")
eixos[0].axvline(dados.median(), color=AZUL, linewidth=2, linestyle="--",
                 label=f"Mediana: {dados.median():.1f} h")
eixos[0].set_title("Tempo de resolução: assimetria à direita")
eixos[0].set_xlabel("horas até o encerramento")
eixos[0].set_ylabel("ocorrências")
eixos[0].legend()

# Direita: boxplot por região, ordenado pela mediana
ordem = dfc.groupby("regiao")["tempo_resolucao_horas"].median().sort_values().index
grupos = [dfc.loc[dfc.regiao == r, "tempo_resolucao_horas"].dropna() for r in ordem]
caixa = eixos[1].boxplot(grupos, labels=ordem, patch_artist=True, showfliers=False,
                         medianprops={"color": TINTA, "linewidth": 2})
for corpo, cor in zip(caixa["boxes"], CATEGORICA):
    corpo.set_facecolor(cor)
    corpo.set_alpha(0.75)
    corpo.set_edgecolor(PAPEL)
eixos[1].set_title("Tempo de resolução por região")
eixos[1].set_ylabel("horas")

plt.tight_layout()
salvar_figura(11, "descritiva-tempo-resolucao")
plt.show()

O histograma mostra a **média à direita da mediana**, a assinatura clássica de uma
distribuição assimétrica. A maioria das ocorrências resolve rápido, e uma minoria de
casos muito demorados puxa a média para cima.

O boxplot já antecipa o achado do **elo 3**: as regiões não se comportam igual. Essa
diferença visual será testada formalmente na seção 2.5.

---
## 2.2 Amostragem e intervalo de confiança

Um Centro de Operações real não processa a base inteira a cada consulta. A pergunta
prática do gestor é: **dá para decidir olhando uma amostra?**

Vamos comparar três estimativas do tempo médio de resolução:

| Estimativa | Como é obtida |
|---|---|
| População | Todas as ocorrências |
| Amostra aleatória simples | 400 sorteadas ao acaso |
| Amostra estratificada | 400 respeitando a proporção de cada região |

A estratificada tende a errar menos quando a variável de interesse difere entre os
estratos, que é exatamente o caso aqui.

### Quantas ocorrências a amostra precisa ter?

Antes de sortear, é preciso **dimensionar**. O tamanho da amostra para uma variável
contínua vem da fórmula:

$$n = \frac{Z^2 \cdot S^2 \cdot N}{ME^2 \cdot (N-1) + Z^2 \cdot S^2}$$

em que `N` é o tamanho da população, `Z` o valor da normal para o nível de confiança
(1,96 para 95%), `S` o desvio padrão obtido numa **amostra-piloto** e `ME` a margem
de erro aceita.

In [ ]:
marcar(12, "Dimensionamento da amostra e comparação entre métodos")

SEMENTE = 570088
base = dfc.dropna(subset=["tempo_resolucao_horas"])
media_populacao = base["tempo_resolucao_horas"].mean()


def formula_amostra_continua(N, S, Z, ME):
    """Tamanho de amostra para variável contínua."""
    n = (Z**2 * S**2 * N) / ((ME**2 * (N - 1)) + (Z**2 * S**2))
    return int(np.ceil(n))


# Amostra-piloto: 10% da base, usada só para estimar o desvio padrão
piloto = base.sample(frac=0.10, random_state=SEMENTE)
S_piloto = piloto["tempo_resolucao_horas"].std(ddof=1)

N_populacao = len(base)
Z_95 = 1.96
MARGEM_ERRO = 1.5          # aceitamos errar até 1,5 hora na estimativa

TAMANHO = formula_amostra_continua(N_populacao, S_piloto, Z_95, MARGEM_ERRO)

print(f"População (N) ................ {N_populacao:,}")
print(f"Desvio padrão do piloto (S) .. {S_piloto:.2f} h   (piloto com {len(piloto)} ocorrências)")
print(f"Nível de confiança (Z) ....... {Z_95}  (95%)")
print(f"Margem de erro (ME) .......... {MARGEM_ERRO} h")
print(f"TAMANHO DA AMOSTRA (n) ....... {TAMANHO}")
print()
print(f"Basta analisar {TAMANHO / N_populacao * 100:.1f}% da base para estimar a média")
print(f"com erro de no máximo {MARGEM_ERRO} h e 95% de confiança.")
print()

# Amostra aleatória simples
simples = base.sample(n=TAMANHO, random_state=SEMENTE)

# Amostra estratificada proporcional por região.
# O train_test_split com o parâmetro stratify é o método do Cap 12: ele garante
# que cada região apareça na amostra na mesma proporção em que aparece na base.
_, estratificada = train_test_split(
    base,
    test_size=TAMANHO,
    random_state=SEMENTE,
    stratify=base["regiao"],
)

comparacao_amostras = pd.DataFrame({
    "n": [len(base), len(simples), len(estratificada)],
    "média (h)": [media_populacao,
                  simples["tempo_resolucao_horas"].mean(),
                  estratificada["tempo_resolucao_horas"].mean()],
}, index=["População", "Amostra simples", "Amostra estratificada"])

comparacao_amostras["erro vs população (h)"] = (
    comparacao_amostras["média (h)"] - media_populacao)
comparacao_amostras["erro relativo (%)"] = (
    comparacao_amostras["erro vs população (h)"] / media_populacao * 100)

comparacao_amostras.round(3)

In [ ]:
marcar(13, "Intervalo de confiança de 95% e Teorema Central do Limite")

# IC de 95% da amostra simples, via erro padrão da média
n = len(simples)
media_amostra = simples["tempo_resolucao_horas"].mean()
erro_padrao = simples["tempo_resolucao_horas"].std(ddof=1) / np.sqrt(n)
t_critico = st.t.ppf(0.975, df=n - 1)
ic_baixo, ic_alto = media_amostra - t_critico * erro_padrao, media_amostra + t_critico * erro_padrao

print(f"Média da população .......... {media_populacao:.2f} h")
print(f"Média da amostra (n={n}) .... {media_amostra:.2f} h")
print(f"Erro padrão ................. {erro_padrao:.3f} h")
print(f"IC 95% ...................... [{ic_baixo:.2f} h , {ic_alto:.2f} h]")
print(f"A população está dentro do intervalo? {'SIM' if ic_baixo <= media_populacao <= ic_alto else 'NAO'}")

# Distribuição de 1.000 médias amostrais: o TCL em ação
rng = np.random.default_rng(SEMENTE)
medias = [base["tempo_resolucao_horas"].sample(n=TAMANHO, random_state=int(rng.integers(1e6))).mean()
          for _ in range(1000)]

fig, eixo = plt.subplots(figsize=(11, 4.2))
eixo.hist(medias, bins=40, color=AZUL, alpha=0.75, edgecolor=PAPEL, linewidth=0.5)
eixo.axvline(media_populacao, color=TERRACOTA, linewidth=2,
             label=f"Média da população: {media_populacao:.2f} h")
eixo.axvspan(ic_baixo, ic_alto, color=OURO, alpha=0.20, label="IC 95% de uma amostra")
eixo.set_title("Teorema Central do Limite: 1.000 médias de amostras de 400 ocorrências")
eixo.set_xlabel("média amostral do tempo de resolução (h)")
eixo.set_ylabel("frequência")
eixo.legend()

plt.tight_layout()
salvar_figura(13, "amostragem-tcl")
plt.show()

**Conclusão da amostragem.** Mesmo com uma variável fortemente assimétrica, a
distribuição das médias amostrais é aproximadamente **normal e centrada no valor real**.
É o Teorema Central do Limite funcionando na prática.

A consequência para a gestão é concreta: **400 ocorrências bastam** para estimar o
tempo médio com margem estreita. O COU não precisa processar dez mil registros para
acompanhar o indicador no painel.

---
## 2.3 ELO 1 — Detecção

> **Hipótese do gestor:** sensores detectam ocorrências mais rápido que o cidadão.

Se isso for verdade, o investimento em IoT se justifica na ponta da detecção.

| | |
|---|---|
| **H₀** | O tempo de resposta é igual entre fontes automáticas e humanas |
| **H₁** | Os tempos são diferentes |
| **Teste** | t de Student para amostras independentes, variâncias desiguais (Welch) |
| **Significância** | α = 0,05 |

Uso a variante de Welch porque os dois grupos têm tamanhos e variâncias diferentes,
e ela não exige a suposição de variâncias iguais.

In [ ]:
marcar(14, "ELO 1 — Teste t: fonte automática x canal humano")

automatica = dfc.loc[dfc.tipo_fonte == "AUTOMATICA", "tempo_resposta_min"].dropna()
cidadao = dfc.loc[dfc.tipo_fonte == "CIDADAO", "tempo_resposta_min"].dropna()

# Método 1: scipy.stats
t_stat, p_valor = st.ttest_ind(automatica, cidadao, equal_var=False, alternative="two-sided")

print(f"Fonte AUTOMATICA  n={len(automatica):>5}  média={automatica.mean():>6.1f} min  mediana={automatica.median():>6.1f} min")
print(f"Fonte CIDADAO     n={len(cidadao):>5}  média={cidadao.mean():>6.1f} min  mediana={cidadao.median():>6.1f} min")
print()
print(f"Diferença absoluta ....... {cidadao.mean() - automatica.mean():.1f} min")
print(f"Ganho relativo ........... {(1 - automatica.mean() / cidadao.mean()) * 100:.1f}%")
print()
print("--- scipy.stats.ttest_ind ---")
print(f"Estatística t ............ {t_stat:.2f}")
print(f"Valor-p .................. {p_valor:.3e}")
print()
print(f"DECISÃO: {'rejeitamos H0' if p_valor < 0.05 else 'não rejeitamos H0'} (α = 0,05)")
print("O ELO 1 FUNCIONA." if p_valor < 0.05 and automatica.mean() < cidadao.mean() else "O ELO 1 NÃO se confirma.")

O `scipy.stats` entrega a estatística t e o valor-p. O **pingouin** entrega o mesmo
teste com informações que a decisão de gestão precisa: o **tamanho do efeito**
(`cohen-d`), o **intervalo de confiança da diferença** e o **poder do teste**.

Poder próximo de 1 significa risco baixo de Erro Tipo II, ou seja, de deixar de
detectar uma diferença que existe de verdade.

In [ ]:
# Método 2: pingouin, com tamanho de efeito e poder
resultado_pg = pg.ttest(x=automatica, y=cidadao, alternative="two-sided",
                        confidence=0.95, correction=True)

d_cohen = col_pg(resultado_pg, "cohen-d", "cohen_d").iloc[0]
poder = resultado_pg["power"].iloc[0]

print("--- pingouin.ttest ---")
print(resultado_pg.round(4).to_string())
print()
print(f"d de Cohen ... {d_cohen:.3f}  "
      f"({'grande' if abs(d_cohen) >= 0.8 else 'médio' if abs(d_cohen) >= 0.5 else 'pequeno'})")
print(f"Poder ........ {poder:.3f}  "
      f"({'risco baixo' if poder > 0.8 else 'atenção'} de Erro Tipo II)")

In [ ]:
marcar(15, "ELO 1 — Distribuição do tempo de resposta por natureza da fonte")

fig, eixos = plt.subplots(1, 2, figsize=(13, 4.8))

# Esquerda: densidades sobrepostas
for serie, rotulo, cor in [(automatica, "Automática (IoT, câmera)", AZUL),
                           (cidadao, "Cidadão (app, portal, 156)", TERRACOTA)]:
    eixos[0].hist(serie, bins=60, range=(0, 400), alpha=0.6, color=cor,
                  label=f"{rotulo}  (média {serie.mean():.0f} min)", density=True)
eixos[0].set_title("Tempo até a equipe chegar ao local")
eixos[0].set_xlabel("minutos")
eixos[0].set_ylabel("densidade")
eixos[0].legend()

# Direita: média por canal específico, ordenada
por_canal = (dfc.groupby("fonte_dado")
               .agg(media=("tempo_resposta_min", "mean"),
                    tipo=("tipo_fonte", "first"))
               .sort_values("media"))
cores = [AZUL if t == "AUTOMATICA" else TERRACOTA for t in por_canal["tipo"]]
barras = eixos[1].barh(por_canal.index, por_canal["media"], color=cores, height=0.6)
for barra, valor in zip(barras, por_canal["media"]):
    eixos[1].text(valor + 1.5, barra.get_y() + barra.get_height() / 2,
                  f"{valor:.0f} min", va="center", color=TINTA, fontsize=9)
eixos[1].set_title("Tempo médio de resposta por canal de origem")
eixos[1].set_xlabel("minutos")
eixos[1].set_xlim(0, por_canal["media"].max() * 1.18)

plt.tight_layout()
salvar_figura(15, "elo1-deteccao")
plt.show()

**Conclusão do elo 1.** A hipótese se confirma com folga. A detecção automática chega
ao local em aproximadamente metade do tempo, e o tamanho do efeito é grande, ou seja,
a diferença não é apenas estatisticamente detectável, é operacionalmente relevante.

**O investimento em sensores entrega o que promete nesta etapa.** A pergunta passa a
ser se esse ganho sobrevive às etapas seguintes.

---
## 2.4 ELO 2 — Despacho

Este é o elo decisivo da análise. Duas perguntas:

**2.4a** O UrbanIQ calcula um `score_prioridade` de 0 a 100 para cada ocorrência.
A central usa esse score para decidir quem é acionado primeiro?

**2.4b** O ganho de tempo do sensor se converte em mais prazos cumpridos?

In [ ]:
marcar(16, "ELO 2a — O score de prioridade influencia o tempo até o despacho?")

elo2 = dfc.dropna(subset=["score_prioridade", "lag_despacho_min"])

r_pearson, p_pearson = st.pearsonr(elo2["score_prioridade"], elo2["lag_despacho_min"])
r_spearman, p_spearman = st.spearmanr(elo2["score_prioridade"], elo2["lag_despacho_min"])

print(f"n = {len(elo2):,} ocorrências")
print()
print(f"Pearson  (relação linear)    r = {r_pearson:+.4f}   p = {p_pearson:.4f}")
print(f"Spearman (relação monótona)  r = {r_spearman:+.4f}   p = {p_spearman:.4f}")
print()
print(f"Variância explicada (r²) ... {r_pearson**2 * 100:.2f}%")
print()
if p_pearson >= 0.05:
    print("DECISÃO: não rejeitamos H0. NÃO HÁ relação detectável.")
    print("O score é calculado, mas NÃO é usado na fila de despacho.")
else:
    print(f"DECISÃO: relação estatisticamente detectável, porém {'fraca' if abs(r_pearson) < 0.3 else 'relevante'}.")

# Faixas de score para ver se existe algum degrau
elo2 = elo2.copy()
elo2["faixa_score"] = pd.cut(elo2["score_prioridade"], [0, 25, 50, 75, 100],
                             labels=["0-25 baixa", "26-50 média", "51-75 alta", "76-100 crítica"])
print()
print("Tempo mediano até o despacho por faixa de prioridade:")
print(elo2.groupby("faixa_score")["lag_despacho_min"].agg(
    n="size", mediana_min="median", media_min="mean").round(1))

In [ ]:
marcar(17, "ELO 2b — A fonte do dado influencia o cumprimento do SLA?")

finalizadas = dfc[dfc["sla_cumprido"].isin(["SIM", "NAO"])]
tabela = pd.crosstab(finalizadas["tipo_fonte"], finalizadas["sla_cumprido"])

qui2, p_qui, gl, esperado = st.chi2_contingency(tabela)

# V de Cramér pela função do scipy (método do Cap 14)
v_cramer = st.contingency.association(tabela, method="cramer")

percentual = (tabela["SIM"] / tabela.sum(axis=1) * 100).round(1)

print("Tabela de contingência:")
print(tabela)
print()
print("% dentro do prazo:")
for indice, valor in percentual.items():
    print(f"  {indice:<12} {valor:.1f}%")
print()
print(f"Qui-quadrado ....... {qui2:.2f}   gl = {gl}")
print(f"Valor-p ............ {p_qui:.4f}")
print(f"V de Cramér ........ {v_cramer:.4f}  (0 = nenhuma associação, 1 = total)")
print()
if p_qui >= 0.05:
    print("DECISÃO: não rejeitamos H0. NÃO HÁ associação.")
    print("O ganho de 45 minutos na detecção NÃO se converte em prazo cumprido.")

In [ ]:
marcar(18, "ELO 2 — Onde o ganho do sensor se perde")

fig, eixos = plt.subplots(1, 2, figsize=(13, 4.8))

# Esquerda: score x lag de despacho, com a reta de tendência
amostra_grafico = elo2.sample(n=min(2500, len(elo2)), random_state=570088)
eixos[0].scatter(amostra_grafico["score_prioridade"], amostra_grafico["lag_despacho_min"],
                 s=9, alpha=0.30, color=ROXO, edgecolors="none")
coef = np.polyfit(elo2["score_prioridade"], elo2["lag_despacho_min"], 1)
linha_x = np.array([elo2["score_prioridade"].min(), elo2["score_prioridade"].max()])
eixos[0].plot(linha_x, np.polyval(coef, linha_x), color=TERRACOTA, linewidth=2,
              label=f"tendência (r = {r_pearson:+.3f})")
eixos[0].set_title("Prioridade não afeta o tempo de despacho")
eixos[0].set_xlabel("score de prioridade (0 a 100)")
eixos[0].set_ylabel("minutos até o acionamento da equipe")
eixos[0].set_ylim(0, elo2["lag_despacho_min"].quantile(0.99))
eixos[0].legend()

# Direita: a cadeia, etapa por etapa
etapas = ["Detecção\n(chegada ao local)", "SLA cumprido"]
valores_auto = [automatica.mean(), percentual.get("AUTOMATICA", 0)]
valores_cid = [cidadao.mean(), percentual.get("CIDADAO", 0)]

# Normaliza para comparar em um mesmo eixo percentual de vantagem
vantagem = [
    (1 - automatica.mean() / cidadao.mean()) * 100,
    percentual.get("AUTOMATICA", 0) - percentual.get("CIDADAO", 0),
]
cores_vantagem = [OLIVA if v > 3 else SUAVE for v in vantagem]
barras = eixos[1].bar(etapas, vantagem, color=cores_vantagem, width=0.5)
for barra, valor in zip(barras, vantagem):
    eixos[1].text(barra.get_x() + barra.get_width() / 2, valor + 1.2,
                  f"{valor:+.1f} p.p." if abs(valor) < 20 else f"{valor:+.0f}%",
                  ha="center", color=TINTA, fontweight="bold")
eixos[1].axhline(0, color=SUAVE, linewidth=1)
eixos[1].set_title("Vantagem da fonte automática sobre o canal humano")
eixos[1].set_ylabel("vantagem")
eixos[1].set_ylim(min(vantagem) - 8, max(vantagem) + 12)

plt.tight_layout()
salvar_figura(18, "elo2-despacho")
plt.show()

### Conclusão do elo 2, o achado central da análise

Os dois testes convergem para o mesmo diagnóstico:

1. O `score_prioridade` **não tem relação detectável** com o tempo até o despacho. A
   central atende por ordem de chegada, e o motor de priorização é ignorado na fila.
2. A fonte do dado **não tem associação** com o cumprimento do SLA.

O gráfico da direita resume a história: a cidade ganha uma vantagem expressiva na
detecção e a **devolve por completo** na etapa seguinte.

**A cadeia quebra no elo 2.** O sensor é rápido, a cidade não é.

---
## 2.5 ELO 3 — Execução

A execução funciona, mas funciona igual para todos os bairros?

| | |
|---|---|
| **H₀** | O tempo de resolução é igual entre as cinco regiões |
| **H₁** | Pelo menos uma região difere das demais |
| **Teste** | ANOVA de um fator, complementada por qui-quadrado sobre o SLA |

A ANOVA compara mais de dois grupos de uma vez. Usar vários testes t aos pares
inflaria a chance de falso positivo, problema conhecido como comparações múltiplas.

In [ ]:
marcar(19, "ELO 3 — ANOVA e desigualdade territorial no cumprimento do SLA")

grupos_regiao = [dfc.loc[dfc.regiao == r, "tempo_resolucao_horas"].dropna()
                 for r in sorted(dfc["regiao"].unique())]

# Teste F pelo scipy
f_stat, p_anova = st.f_oneway(*grupos_regiao)

# Kruskal-Wallis: versão não paramétrica, robusta à assimetria observada em 2.1
h_stat, p_kruskal = st.kruskal(*grupos_regiao)

print("--- scipy.stats ---")
print(f"ANOVA (teste F)  F = {f_stat:.2f}    p = {p_anova:.3e}")
print(f"Kruskal-Wallis   H = {h_stat:.2f}    p = {p_kruskal:.3e}")
print(f"DECISÃO: {'rejeitamos H0, as regiões DIFEREM' if p_anova < 0.05 else 'não rejeitamos H0'}")
print()

# Mesma ANOVA pelo pingouin, que traz o tamanho de efeito (np2)
anova_pg = pg.anova(dv="tempo_resolucao_horas", between="regiao",
                    data=dfc.dropna(subset=["tempo_resolucao_horas"]), detailed=True)
print("--- pingouin.anova ---")
print(anova_pg.round(4).to_string(index=False))
print()

finalizadas_regiao = dfc[dfc["sla_cumprido"].isin(["SIM", "NAO"])]
tabela_regiao = pd.crosstab(finalizadas_regiao["regiao"], finalizadas_regiao["sla_cumprido"])
qui2_r, p_r, gl_r, _ = st.chi2_contingency(tabela_regiao)
v_cramer_r = st.contingency.association(tabela_regiao, method="cramer")

painel_regiao = pd.DataFrame({
    "ocorrências": dfc.groupby("regiao").size(),
    "tempo_resolução_mediano_h": dfc.groupby("regiao")["tempo_resolucao_horas"].median(),
    "tempo_resposta_médio_min": dfc.groupby("regiao")["tempo_resposta_min"].mean(),
    "sla_cumprido_%": (tabela_regiao["SIM"] / tabela_regiao.sum(axis=1) * 100),
    "satisfação_média": dfc.groupby("regiao")["satisfacao_cidadao"].mean(),
}).sort_values("sla_cumprido_%", ascending=False)

print(f"Qui-quadrado região x SLA:  {qui2_r:.2f}   p = {p_r:.3e}   V de Cramér = {v_cramer_r:.3f}")
print()
amplitude = painel_regiao["sla_cumprido_%"].max() - painel_regiao["sla_cumprido_%"].min()
print(f"Amplitude entre a melhor e a pior região: {amplitude:.1f} pontos percentuais")
painel_regiao.round(2)

In [ ]:
marcar(20, "ELO 3 — Desempenho por região")

fig, eixos = plt.subplots(1, 2, figsize=(13, 4.8))

ordenado = painel_regiao.sort_values("sla_cumprido_%")
media_cidade = (finalizadas_regiao["sla_cumprido"] == "SIM").mean() * 100

# Esquerda: SLA cumprido por região, destacando os extremos
cores_sla = [TERRACOTA if v < media_cidade - 8 else OLIVA if v > media_cidade + 8 else SUAVE
             for v in ordenado["sla_cumprido_%"]]
barras = eixos[0].barh(ordenado.index, ordenado["sla_cumprido_%"], color=cores_sla, height=0.6)
for barra, valor in zip(barras, ordenado["sla_cumprido_%"]):
    eixos[0].text(valor + 1, barra.get_y() + barra.get_height() / 2,
                  f"{valor:.1f}%", va="center", color=TINTA, fontsize=9)
eixos[0].axvline(media_cidade, color=TINTA, linestyle="--", linewidth=1.5,
                 label=f"média da cidade: {media_cidade:.1f}%")
eixos[0].set_title("Prazo cumprido por região")
eixos[0].set_xlabel("% das ocorrências dentro do SLA")
eixos[0].set_xlim(0, 100)
eixos[0].legend(loc="lower right")

# Direita: SLA x satisfação, evidenciando que o cidadão percebe
eixos[1].scatter(painel_regiao["sla_cumprido_%"], painel_regiao["satisfação_média"],
                 s=140, color=OURO, edgecolors=TINTA, linewidths=1, zorder=3)
for regiao, linha in painel_regiao.iterrows():
    eixos[1].annotate(regiao, (linha["sla_cumprido_%"], linha["satisfação_média"]),
                      textcoords="offset points", xytext=(0, 11),
                      ha="center", fontsize=9, color=TINTA)
eixos[1].set_title("Onde o prazo é cumprido, o cidadão avalia melhor")
eixos[1].set_xlabel("% dentro do SLA")
eixos[1].set_ylabel("satisfação média (1 a 5)")

plt.tight_layout()
salvar_figura(20, "elo3-execucao")
plt.show()

**Conclusão do elo 3.** A ANOVA e o Kruskal-Wallis concordam: as regiões diferem, e
a diferença não é ruído amostral.

A amplitude entre a melhor e a pior região é grande o bastante para ser tratada como
**desigualdade territorial no serviço público**, não como variação operacional. Duas
pessoas com o mesmo problema recebem serviços de qualidade muito diferente dependendo
de onde moram.

O gráfico da direita antecipa o elo 4: a satisfação acompanha o cumprimento do prazo.

---
## 2.6 ELO 4 — Percepção do cidadão

O que determina a nota que o cidadão dá: o **tempo absoluto** de atendimento ou a
**promessa cumprida**?

A diferença não é acadêmica. Ela decide a recomendação: "seja mais rápido" ou
"prometa prazos realistas".

In [ ]:
marcar(21, "ELO 4 — Matriz de correlação das variáveis da cadeia")

variaveis_cadeia = [
    "lag_despacho_min", "tempo_resposta_min", "tempo_resolucao_horas",
    "razao_sla", "score_prioridade", "qtd_pessoas_afetadas",
    "custo_operacional_reais", "satisfacao_cidadao",
]
matriz = dfc[variaveis_cadeia].corr(method="pearson")

fig, eixo = plt.subplots(figsize=(9.5, 7.5))
imagem = eixo.imshow(matriz, cmap="RdBu_r", vmin=-1, vmax=1)

eixo.set_xticks(range(len(variaveis_cadeia)))
eixo.set_yticks(range(len(variaveis_cadeia)))
eixo.set_xticklabels(variaveis_cadeia, rotation=45, ha="right", fontsize=9)
eixo.set_yticklabels(variaveis_cadeia, fontsize=9)
eixo.grid(False)

for i in range(len(variaveis_cadeia)):
    for j in range(len(variaveis_cadeia)):
        valor = matriz.iloc[i, j]
        eixo.text(j, i, f"{valor:.2f}", ha="center", va="center", fontsize=8,
                  color="white" if abs(valor) > 0.55 else TINTA)

eixo.set_title("Correlação de Pearson entre as variáveis da cadeia de valor")
fig.colorbar(imagem, ax=eixo, shrink=0.8, label="coeficiente r")

plt.tight_layout()
salvar_figura(21, "elo4-matriz-correlacao")
plt.show()

print()
print("Correlações com a satisfação do cidadão, da mais forte para a mais fraca:")
print(matriz["satisfacao_cidadao"].drop("satisfacao_cidadao")
        .sort_values(key=abs, ascending=False).round(4))

### Os três coeficientes e o teste de significância

Pearson mede relação **linear**, Spearman e Kendall medem relação **monótona**, ou seja,
sobrevivem a relações curvas e são menos sensíveis a valores extremos. Comparar os três
é uma checagem de robustez: se os três concordam, a relação é sólida.

O `pingouin.pairwise_corr` acrescenta o que falta para decidir: intervalo de confiança,
valor-p e poder de cada correlação.

In [ ]:
# Os três coeficientes lado a lado, com leitura facilitada por mapa de cores
for metodo in ["pearson", "spearman", "kendall"]:
    serie = (dfc[variaveis_cadeia].corr(method=metodo)["satisfacao_cidadao"]
               .drop("satisfacao_cidadao").sort_values(key=abs, ascending=False))
    print(f"--- {metodo.upper()} vs satisfacao_cidadao ---")
    print(serie.round(4).to_string())
    print()

# Matriz de Pearson com gradiente de cor (método do Cap 14)
dfc[variaveis_cadeia].corr().style.background_gradient(cmap="coolwarm").format("{:.2f}")

In [ ]:
# Teste de significância das correlações com a satisfação
teste_corr = pg.pairwise_corr(
    dfc[variaveis_cadeia],
    columns=[["satisfacao_cidadao"], ["razao_sla", "tempo_resposta_min",
                                      "tempo_resolucao_horas", "lag_despacho_min",
                                      "score_prioridade"]],
    method="pearson")

resumo_corr = pd.DataFrame({
    "X": teste_corr["X"],
    "Y": teste_corr["Y"],
    "n": teste_corr["n"],
    "r": teste_corr["r"].round(4),
    "IC95%": col_pg(teste_corr, "CI95%", "CI95"),
    "valor_p": col_pg(teste_corr, "p-unc", "p_unc").apply(lambda v: f"{v:.2e}"),
    "poder": teste_corr["power"].round(3),
})
print(resumo_corr.to_string(index=False))
print()
print("Poder igual a 1 indica que o tamanho da amostra é mais que suficiente")
print("para detectar a relação, se ela existir.")

In [ ]:
marcar(22, "ELO 4 — Regressão: o que pesa mais, o tempo absoluto ou o prazo quebrado?")

comparativo = []
for variavel in ["tempo_resposta_min", "tempo_resolucao_horas", "razao_sla", "lag_despacho_min"]:
    par = dfc[[variavel, "satisfacao_cidadao"]].dropna()
    r, p = st.pearsonr(par[variavel], par["satisfacao_cidadao"])
    rho, _ = st.spearmanr(par[variavel], par["satisfacao_cidadao"])
    comparativo.append({"variável": variavel, "n": len(par), "pearson_r": r,
                        "spearman_rho": rho, "r²_%": r**2 * 100, "valor_p": p})

comparativo = pd.DataFrame(comparativo).sort_values("pearson_r", key=abs, ascending=False)
print(comparativo.to_string(index=False))
print()

# Regressão linear sobre o vencedor
vencedora = comparativo.iloc[0]["variável"]
par = dfc[[vencedora, "satisfacao_cidadao"]].dropna()
regressao = st.linregress(par[vencedora], par["satisfacao_cidadao"])

print(f"Regressão linear: satisfacao_cidadao ~ {vencedora}")
print(f"  coeficiente angular ... {regressao.slope:+.4f}")
print(f"  intercepto ............ {regressao.intercept:.4f}")
print(f"  r² .................... {regressao.rvalue**2:.4f}")
print(f"  valor-p ............... {regressao.pvalue:.3e}")
print()
print(f"Leitura: cada aumento de 1 unidade em {vencedora} reduz a nota")
print(f"do cidadão em {abs(regressao.slope):.4f} ponto, na média.")

fig, eixo = plt.subplots(figsize=(11, 4.6))
faixas = pd.cut(par[vencedora], bins=12)
agrupado = par.groupby(faixas)["satisfacao_cidadao"].agg(["mean", "size"])
centros = [intervalo.mid for intervalo in agrupado.index]

eixo.scatter(centros, agrupado["mean"], s=agrupado["size"] / 12, color=OURO,
             edgecolors=TINTA, linewidths=0.8, zorder=3, label="satisfação média da faixa")
linha_x = np.linspace(par[vencedora].min(), par[vencedora].max(), 100)
eixo.plot(linha_x, regressao.intercept + regressao.slope * linha_x,
          color=TERRACOTA, linewidth=2, label=f"regressão (r² = {regressao.rvalue**2:.3f})")
eixo.axvline(1.0, color=SUAVE, linestyle="--", linewidth=1.5)
eixo.text(1.02, eixo.get_ylim()[1] * 0.97, "prazo prometido", fontsize=9,
          color=SUAVE, va="top")
eixo.set_title("A satisfação cai conforme o prazo prometido é estourado")
eixo.set_xlabel(f"{vencedora}  (1,0 = prazo cumprido no limite)")
eixo.set_ylabel("satisfação média (1 a 5)")
eixo.legend()

plt.tight_layout()
salvar_figura(22, "elo4-regressao")
plt.show()

**Conclusão do elo 4.** A `razao_sla` correlaciona mais forte com a satisfação do que
o tempo absoluto. O tamanho das bolhas no gráfico mostra onde está o volume de casos,
o que evita a leitura enganosa de faixas com poucos registros.

A tradução para a gestão é direta: **o cidadão não pune a demora, pune a promessa
quebrada.** Uma ocorrência de pavimentação resolvida em 60 horas com prazo de 72
satisfaz mais que uma de trânsito resolvida em 8 horas com prazo de 6.

---
## 2.7 Limites, derivadas e integrais aplicados

Três ferramentas do cálculo, cada uma respondendo a uma pergunta que a estatística
descritiva não alcança.

| Ferramenta | Pergunta que responde |
|---|---|
| **Derivada** | A demanda está acelerando? Quando? |
| **Integral** | Qual é o passivo acumulado que a cidade carrega? |
| **Limite** | Contratar mais equipes ainda compensa? |

### 2.7.1 Derivada — a demanda está acelerando?

O volume mensal diz **quanto** se tem. A derivada diz **para onde está indo**.

Um bairro com 200 ocorrências estáveis é menos urgente que um com 80 e aceleração
positiva.

Vamos obter a derivada de **duas formas**, e comparar:

| Abordagem | Ferramenta | Serve para |
|---|---|---|
| **Numérica** | `np.gradient` (diferenças finitas) | Medir a variação real, mês a mês |
| **Simbólica** | `sympy.diff` sobre uma função ajustada | Obter a expressão de f'(x) e achar máximos e mínimos |

A simbólica é a do Cap 9: transforma a variável em símbolo com `Symbol`, deriva com
`diff` e converte de volta para função Python com `lambdify`.

In [ ]:
marcar(23, "Derivada — taxa de variação do volume mensal de ocorrências")

serie_mensal = (dfc.set_index("dt_abertura")
                   .resample("MS")
                   .size()
                   .rename("ocorrencias"))

# Primeira derivada: variação de ocorrências por mês
derivada = np.gradient(serie_mensal.values.astype(float))

fig, eixos = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                          gridspec_kw={"height_ratios": [1.5, 1]})

eixos[0].plot(serie_mensal.index, serie_mensal.values, color=AZUL, linewidth=2,
              marker="o", markersize=4)
eixos[0].fill_between(serie_mensal.index, serie_mensal.values, color=AZUL, alpha=0.12)
eixos[0].set_title("Volume mensal de ocorrências e sua taxa de variação")
eixos[0].set_ylabel("ocorrências no mês")

cores_derivada = [OLIVA if v <= 0 else TERRACOTA for v in derivada]
eixos[1].bar(serie_mensal.index, derivada, color=cores_derivada, width=20)
eixos[1].axhline(0, color=TINTA, linewidth=1)
eixos[1].set_ylabel("d(ocorrências)/d(mês)")
eixos[1].set_xlabel("mês")

plt.tight_layout()
salvar_figura(23, "calculo-derivada")
plt.show()

pico = int(np.argmax(derivada))
print(f"Maior aceleração da demanda: {serie_mensal.index[pico]:%b/%Y} "
      f"({derivada[pico]:+.0f} ocorrências/mês)")
print(f"Maior desaceleração:         {serie_mensal.index[int(np.argmin(derivada))]:%b/%Y} "
      f"({derivada.min():+.0f} ocorrências/mês)")
print()
print("Uso operacional: a barra vermelha antecipa o pico. Um COU que monitora a")
print("derivada mobiliza equipes ANTES do volume estourar, não depois.")

In [ ]:
# --- Derivada SIMBÓLICA com SymPy (método do Cap 9) ---

# 1. Ajusta um polinômio de grau 3 à série mensal de ocorrências
meses = np.arange(len(serie_mensal), dtype=float)
coeficientes = np.polyfit(meses, serie_mensal.values.astype(float), 3)

# 2. Transforma a variável em SÍMBOLO e monta a função
x = smp.Symbol("x")
# Coeficientes arredondados: sem isso o SymPy imprime frações gigantes
funcao = sum(smp.Float(round(float(c), 4)) * x**(3 - i)
             for i, c in enumerate(coeficientes))

print("f(x), volume de ocorrências em função do mês:")
smp.pprint(smp.expand(funcao))
print()

# 3. Deriva simbolicamente
funcao_derivada = smp.diff(funcao, x)
print("f'(x), taxa de variação:")
smp.pprint(smp.expand(funcao_derivada))
print()

# 4. Pontos críticos: onde f'(x) = 0, a demanda para de subir ou de cair
criticos = smp.solve(funcao_derivada, x)
reais = [float(c) for c in criticos if c.is_real and 0 <= float(c) <= len(meses) - 1]
for ponto in reais:
    mes_critico = serie_mensal.index[int(round(ponto))]
    segunda = float(smp.diff(funcao, x, 2).subs(x, ponto))
    tipo = "MÁXIMO (pico da demanda)" if segunda < 0 else "MÍNIMO (vale da demanda)"
    print(f"Ponto crítico em x = {ponto:.1f}  ->  {mes_critico:%b/%Y}  ->  {tipo}")

# 5. lambdify converte a expressão simbólica em função Python utilizável
f_numerica = smp.lambdify(x, funcao, "numpy")
f_linha = smp.lambdify(x, funcao_derivada, "numpy")

fig, eixos = plt.subplots(1, 2, figsize=(13, 4.2))
eixos[0].plot(meses, serie_mensal.values, "o", color=SUAVE, markersize=4, label="observado")
eixos[0].plot(meses, f_numerica(meses), color=AZUL, linewidth=2, label="f(x) ajustada")
eixos[0].set_title("Função ajustada ao volume mensal")
eixos[0].set_xlabel("mês (x)")
eixos[0].set_ylabel("ocorrências")
eixos[0].legend()

eixos[1].plot(meses, f_linha(meses), color=TERRACOTA, linewidth=2, label="f'(x) simbólica")
eixos[1].plot(meses, derivada, "o--", color=OLIVA, markersize=4, alpha=0.7,
              label="np.gradient (numérica)")
eixos[1].axhline(0, color=TINTA, linewidth=1)
for ponto in reais:
    eixos[1].axvline(ponto, color=SUAVE, linestyle=":", linewidth=1.5)
eixos[1].set_title("Derivada: simbólica x numérica")
eixos[1].set_xlabel("mês (x)")
eixos[1].set_ylabel("d(ocorrências)/d(mês)")
eixos[1].legend()

plt.tight_layout()
salvar_figura(23, "calculo-derivada-sympy")
plt.show()

### 2.7.2 Integral — o passivo urbano acumulado

O backlog diário informa quantas ocorrências estão abertas hoje. A **área sob essa
curva** informa algo diferente e mais grave: quanto tempo total a cidade passou com
problemas em aberto.

A unidade resultante é **ocorrência-dia**, que traduzo como exposição acumulada da
população ao risco. Um KPI que não existe no painel atual e que um gestor entende
imediatamente.

A integral é aproximada pela regra dos trapézios.

In [ ]:
marcar(24, "Integral — backlog acumulado e exposição da população ao risco")

# Backlog diário: aberturas menos encerramentos, acumulado no tempo
aberturas = dfc.set_index("dt_abertura").resample("D").size()
encerramentos = dfc.dropna(subset=["dt_encerramento"]).set_index("dt_encerramento").resample("D").size()

calendario = pd.date_range(dfc["dt_abertura"].min().normalize(),
                           dfc["dt_abertura"].max().normalize(), freq="D")
fluxo = pd.DataFrame(index=calendario)
fluxo["abertas"] = aberturas.reindex(calendario, fill_value=0)
fluxo["encerradas"] = encerramentos.reindex(calendario, fill_value=0)
fluxo["backlog"] = (fluxo["abertas"] - fluxo["encerradas"]).cumsum()

dias = np.arange(len(fluxo), dtype=float)
integrar = np.trapezoid if hasattr(np, "trapezoid") else np.trapz   # numpy 2 renomeou trapz
area_total = integrar(fluxo["backlog"].values, dias)

fig, eixo = plt.subplots(figsize=(12, 4.6))
eixo.plot(fluxo.index, fluxo["backlog"], color=TERRACOTA, linewidth=1.6)
eixo.fill_between(fluxo.index, fluxo["backlog"], color=TERRACOTA, alpha=0.18)
eixo.axhline(0, color=TINTA, linewidth=1)
eixo.set_title("Backlog de ocorrências em aberto — a área é o passivo acumulado")
eixo.set_ylabel("ocorrências em aberto")
eixo.set_xlabel("data")
eixo.annotate(f"∫ backlog dt ≈ {area_total:,.0f} ocorrência-dia",
              xy=(0.99, 0.06), xycoords="axes fraction", ha="right",
              fontsize=11, color=TINTA, fontweight="bold")

plt.tight_layout()
salvar_figura(24, "calculo-integral")
plt.show()

print(f"Backlog médio ................. {fluxo['backlog'].mean():,.0f} ocorrências")
print(f"Backlog máximo ................ {fluxo['backlog'].max():,.0f} ocorrências")
print(f"Integral (área sob a curva) ... {area_total:,.0f} ocorrência-dia")
print(f"Período analisado ............. {len(fluxo)} dias")

### Os três métodos de integração, comparados

O Cap 10 apresenta a área sob a curva por dois caminhos: a **soma dos retângulos**,
que é a definição intuitiva, e a **integral simbólica** com `sympy.integrate`, que é
a forma exata. Vamos calcular pelos três e comparar os resultados.

In [ ]:
# --- Método 1: soma dos retângulos (definição do Cap 10) ---
N_RETANGULOS = 24
minimo, maximo = 0, len(fluxo) - 1
dx = (maximo - minimo) / N_RETANGULOS

eixo_x = np.arange(minimo, maximo + dx, dx)
eixo_y = np.interp(eixo_x, np.arange(len(fluxo)), fluxo["backlog"].values)

fx_dx = [altura * dx for altura in eixo_y]
area_retangulos = sum(fx_dx)

# --- Método 2: regra dos trapézios (numérica, já calculada acima) ---
# area_total

# --- Método 3: integral SIMBÓLICA com SymPy ---
coef_backlog = np.polyfit(np.arange(len(fluxo), dtype=float),
                          fluxo["backlog"].values.astype(float), 4)
t = smp.Symbol("t")
funcao_backlog = sum(float(c) * t**(4 - i) for i, c in enumerate(coef_backlog))

integral_indefinida = smp.integrate(funcao_backlog, t)
print("Integral indefinida de B(t):")
smp.pprint(smp.expand(integral_indefinida))
print()

area_simbolica = float(smp.integrate(funcao_backlog, (t, minimo, maximo)))

print(f"Soma dos retângulos (n={N_RETANGULOS}) ... {area_retangulos:>12,.0f} ocorrência-dia")
print(f"Regra dos trapézios ............... {area_total:>12,.0f} ocorrência-dia")
print(f"Integral simbólica (SymPy) ........ {area_simbolica:>12,.0f} ocorrência-dia")
print()
print(f"Diferença retângulos x trapézios .. {abs(area_retangulos - area_total) / area_total * 100:.2f}%")
print(f"Diferença simbólica x trapézios ... {abs(area_simbolica - area_total) / area_total * 100:.2f}%")
print()
print("Os três convergem. A soma dos retângulos superestima ou subestima conforme a")
print("curva sobe ou desce no intervalo; quanto maior n, menor o erro.")

# Visualização dos retângulos sob a curva, como no Cap 10
fig, eixo = plt.subplots(figsize=(12, 4.2))
eixo.bar(eixo_x, eixo_y, width=dx * 0.92, color=ROXO, alpha=0.35,
         align="edge", label=f"{N_RETANGULOS} retângulos")
eixo.plot(np.arange(len(fluxo)), fluxo["backlog"].values, color=TERRACOTA,
          linewidth=1.6, label="backlog observado")
eixo.set_title("Área sob a curva do backlog pela soma dos retângulos")
eixo.set_xlabel("dias desde o início do período")
eixo.set_ylabel("ocorrências em aberto")
eixo.legend()

plt.tight_layout()
salvar_figura(24, "calculo-integral-retangulos")
plt.show()

### 2.7.3 Limite — contratar mais equipes ainda compensa?

Modelo simples de fila com `n` equipes, em que `λ` é a taxa de chegada e `μ` a
capacidade de atendimento por equipe:

$$W(n) = \frac{1}{n\mu - \lambda}, \quad n\mu > \lambda$$

Três perguntas do cálculo, todas resolvidas com **SymPy**, como no Cap 9:

| Ferramenta | Pergunta | Comando |
|---|---|---|
| **Limite ao infinito** | Equipes infinitas zeram a fila? | `smp.limit(W, n, smp.oo)` |
| **Limite lateral** | O que acontece ao chegar perto da capacidade mínima? | `smp.limit(W, n, n_crítico, '+')` |
| **Derivada** | Quanto cada equipe adicional economiza? | `smp.diff(W, n)` |

O ponto de saturação é onde a derivada fica pequena demais para justificar o custo.
É a diferença entre "contratar resolve" e "contratar ainda resolve".

In [ ]:
marcar(25, "Limite — ponto de saturação do ganho por equipe adicional")

EQUIPES_ATUAIS = dfc["equipe_acionada"].nunique()

# λ: ocorrências que chegam por dia
lambda_chegada = len(dfc) / len(calendario)

# μ: capacidade REAL de uma equipe, medida pela vazão observada na base.
# Não uso 24/tempo_medio porque isso trataria a equipe como atendente serial,
# de uma ocorrência por vez. Uma guarnição toca vários chamados em paralelo,
# então a única medida honesta é o que elas de fato entregaram.
finalizadas_total = dfc["dt_encerramento"].notna().sum()
mu_capacidade = finalizadas_total / (EQUIPES_ATUAIS * len(calendario))

utilizacao = lambda_chegada / (EQUIPES_ATUAIS * mu_capacidade)
n_minimo = int(np.ceil(lambda_chegada / mu_capacidade))

print(f"Equipes em operação ................... {EQUIPES_ATUAIS}")
print(f"λ  chegadas por dia ................... {lambda_chegada:.2f}")
print(f"μ  vazão observada por equipe/dia ..... {mu_capacidade:.2f}")
print(f"Capacidade total atual (n·μ) .......... {EQUIPES_ATUAIS * mu_capacidade:.2f}")
print(f"Utilização do sistema (λ / n·μ) ....... {utilizacao:.1%}")
print()
if utilizacao >= 1:
    print(f"ALERTA: a utilização passou de 100%. Chega mais do que sai, e por isso")
    print(f"o backlog da captura 24 cresce sem parar. São necessárias no mínimo")
    print(f"{n_minimo} equipes só para ESTABILIZAR a fila, contra as {EQUIPES_ATUAIS} atuais.")
    print()

# --- Modelo simbólico com SymPy (método dos Caps 9 e 10) ---
n = smp.Symbol("n", positive=True)

# λ e μ entram como racionais exatos. Com float, o limite lateral devolve um
# número enorme em vez de infinito, por erro de arredondamento.
mu_s = smp.Rational(str(round(mu_capacidade, 6)))
lam_s = smp.Rational(str(round(lambda_chegada, 6)))

W = 1 / (n * mu_s - lam_s)                        # espera em dias

print("W(n), tempo de espera em função do número de equipes:")
smp.pprint(smp.N(W, 4))
print()

# LIMITE quando n tende ao infinito
limite_infinito = smp.limit(W, n, smp.oo)
print(f"lim(n->∞) W(n) = {limite_infinito}")
print("  Com equipes infinitas a fila desaparece. Matematicamente verdadeiro,")
print("  economicamente inútil: o custo também tende ao infinito.")
print()

# LIMITE quando n tende ao ponto crítico pela direita: a fila explode
n_critico_s = lam_s / mu_s
limite_critico = smp.limit(W, n, n_critico_s, "+")
print(f"lim(n->{float(n_critico_s):.2f}+) W(n) = {limite_critico}")
print("  Ao se aproximar da capacidade mínima por cima, a espera tende ao infinito.")
print("  É a assíntota vertical do sistema.")
print()

# DERIVADA: ganho marginal de cada equipe adicional
dW = smp.diff(W, n)
print("W'(n), ganho marginal por equipe adicional:")
# Versão só para exibição, com floats arredondados (a matemática usa a racional)
smp.pprint(-smp.Float(round(mu_capacidade, 4))
           / (smp.Float(round(mu_capacidade, 4)) * n
              - smp.Float(round(lambda_chegada, 4)))**2)
print()

# lambdify converte as expressões simbólicas em funções Python
espera_h = smp.lambdify(n, W * 24, "numpy")        # em horas
ganho_h = smp.lambdify(n, -dW * 24, "numpy")      # horas economizadas

# A curva só existe onde n·μ > λ, ou seja, a partir de n_minimo
n_equipes = np.arange(n_minimo, n_minimo + 16)
tempos = espera_h(n_equipes.astype(float))
ganho_marginal = ganho_h(n_equipes.astype(float))

fig, eixos = plt.subplots(1, 2, figsize=(13, 4.6))

eixos[0].plot(n_equipes, tempos, color=AZUL, linewidth=2, marker="o", markersize=4)
eixos[0].axvline(n_minimo, color=TERRACOTA, linestyle="--", linewidth=1.5,
                 label=f"mínimo para estabilizar: {n_minimo} equipes")
eixos[0].axhline(0, color=SUAVE, linewidth=1)
eixos[0].set_title("Tempo de espera em função do número de equipes")
eixos[0].set_xlabel("número de equipes (n)")
eixos[0].set_ylabel("espera estimada (horas)")
eixos[0].set_ylim(0, np.nanmax(tempos[tempos < np.inf]) * 0.6)
eixos[0].legend()

cores_ganho = [OLIVA if g and g > 0.5 else SUAVE for g in ganho_marginal]
eixos[1].bar(n_equipes, ganho_marginal, color=cores_ganho, width=0.6)
eixos[1].axhline(0.5, color=TERRACOTA, linestyle="--", linewidth=1.5,
                 label="limiar de 0,5 h de ganho")
eixos[1].set_title("Ganho marginal de cada equipe adicional")
eixos[1].set_xlabel("número de equipes (n)")
eixos[1].set_ylabel("horas economizadas pela n-ésima equipe")
eixos[1].set_ylim(0, np.nanmax(ganho_marginal[1:]) * 1.1)
eixos[1].legend()

plt.tight_layout()
salvar_figura(25, "calculo-limite")
plt.show()

saturacao = next((int(k) for k, g in zip(n_equipes, ganho_marginal) if g < 0.5), None)
if saturacao:
    print(f"Ponto de saturação: a partir de {saturacao} equipes, W'(n) indica ganho")
    print(f"menor que 0,5 h por equipe. Contratar além disso deixa de compensar.")

---
## 2.8 Armadilhas estatísticas

Três erros que um relatório apressado cometeria com esta mesma base. Reconhecê-los
vale mais que somar mais um teste.

In [ ]:
marcar(26, "Armadilha 1 — correlações tautológicas")

tautologicas = [
    ("criticidade_sensor", "score_prioridade", "o score é CALCULADO a partir da criticidade"),
    ("sla_horas_previsto", "tempo_resolucao_horas", "categoria com prazo maior demora mais, por definição"),
    ("tempo_resolucao_horas", "razao_sla", "a razão é o tempo dividido pelo prazo"),
    ("qtd_pessoas_afetadas", "densidade_hab_km2", "bairro denso afeta mais gente, por construção"),
]

print(f"{'par de variáveis':<52} {'r':>7}   por que NÃO vale")
print("-" * 118)
for a, b, motivo in tautologicas:
    par = dfc[[a, b]].dropna()
    r, _ = st.pearsonr(par[a], par[b])
    print(f"{a + ' x ' + b:<52} {r:>+7.3f}   {motivo}")

print()
print("Todas têm r alto e nenhuma informa nada. São identidades matemáticas,")
print("não descobertas. Foram descartadas da análise por esse motivo.")

In [ ]:
marcar(27, "Armadilha 2 — falácia ecológica")

# Mesma relação, medida em dois níveis de agregação diferentes
individual = dfc[["custo_operacional_reais", "satisfacao_cidadao"]].dropna()
r_individual, p_individual = st.pearsonr(individual["custo_operacional_reais"],
                                            individual["satisfacao_cidadao"])

# "Não Informado" não é um bairro real, sai da análise agregada
por_bairro = (dfc[dfc["bairro"] != "Não Informado"]
              .groupby("bairro")
              .agg(custo=("custo_operacional_reais", "mean"),
                   satisfacao=("satisfacao_cidadao", "mean"))
              .dropna())
r_agregado, p_agregado = st.pearsonr(por_bairro["custo"], por_bairro["satisfacao"])

print(f"Nível OCORRÊNCIA (n = {len(individual):,})   r = {r_individual:+.3f}   p = {p_individual:.2e}")
print(f"Nível BAIRRO     (n = {len(por_bairro):,})       r = {r_agregado:+.3f}   p = {p_agregado:.2e}")
print()
print(f"A mesma relação salta de {abs(r_individual):.3f} para {abs(r_agregado):.3f} apenas por agregar.")
print()
print("Por quê: agregar elimina a variação individual e deixa só a tendência entre")
print("grupos. A correlação infla artificialmente. Concluir algo sobre o cidadão a")
print("partir do dado do bairro é a FALÁCIA ECOLÓGICA.")

fig, eixos = plt.subplots(1, 2, figsize=(13, 4.4))

amostra_ind = individual.sample(n=min(2500, len(individual)), random_state=570088)
eixos[0].scatter(amostra_ind["custo_operacional_reais"], amostra_ind["satisfacao_cidadao"],
                 s=8, alpha=0.20, color=SUAVE, edgecolors="none")
eixos[0].set_title(f"Por ocorrência: r = {r_individual:+.3f} (irrelevante)")
eixos[0].set_xlabel("custo operacional (R$)")
eixos[0].set_ylabel("satisfação (1 a 5)")

eixos[1].scatter(por_bairro["custo"], por_bairro["satisfacao"], s=130, color=OURO,
                 edgecolors=TINTA, linewidths=1, zorder=3)
coef_b = np.polyfit(por_bairro["custo"], por_bairro["satisfacao"], 1)
linha_b = np.array([por_bairro["custo"].min(), por_bairro["custo"].max()])
eixos[1].plot(linha_b, np.polyval(coef_b, linha_b), color=TERRACOTA, linewidth=2)
eixos[1].set_title(f"Por bairro: r = {r_agregado:+.3f} (quase perfeito)")
eixos[1].set_xlabel("custo operacional médio do bairro (R$)")
eixos[1].set_ylabel("satisfação média do bairro")

plt.tight_layout()
salvar_figura(27, "armadilha-falacia-ecologica")
plt.show()

In [ ]:
marcar(28, "Armadilha 3 — correlação espúria por confundimento")

# Temperatura parece explicar o volume... no agregado mensal
diario = pd.DataFrame({
    "volume": dfc.set_index("dt_abertura").resample("D").size(),
    "temperatura": dfc.set_index("dt_abertura")["temperatura_c"].resample("D").max(),
    "chuva": dfc.set_index("dt_abertura")["chuva_mm"].resample("D").max(),
}).dropna()

mensal = diario.resample("MS").mean()

r_dia, p_dia = st.pearsonr(diario["temperatura"], diario["volume"])
r_mes, p_mes = st.pearsonr(mensal["temperatura"], mensal["volume"])
r_tc, _ = st.pearsonr(diario["temperatura"], diario["chuva"])

# Correlação parcial: efeito da temperatura CONTROLANDO a chuva
residuo_volume = diario["volume"] - np.polyval(
    np.polyfit(diario["chuva"], diario["volume"], 1), diario["chuva"])
residuo_temperatura = diario["temperatura"] - np.polyval(
    np.polyfit(diario["chuva"], diario["temperatura"], 1), diario["chuva"])
r_parcial, p_parcial = st.pearsonr(residuo_temperatura, residuo_volume)

print(f"Temperatura x volume, por DIA .................. r = {r_dia:+.3f}  p = {p_dia:.2e}")
print(f"Temperatura x volume, por MÊS .................. r = {r_mes:+.3f}  p = {p_mes:.2e}  <- parece forte")
print()
print(f"Mas temperatura x chuva ........................ r = {r_tc:+.3f}")
print("As duas são sazonais. A chuva é a variável de confundimento.")
print()
print(f"Correlação PARCIAL, controlando a chuva ........ r = {r_parcial:+.3f}  p = {p_parcial:.2e}")
print()
queda = (1 - abs(r_parcial) / abs(r_mes)) * 100 if r_mes else 0
print(f"O efeito aparente da temperatura cai {queda:.0f}% ao controlar a chuva.")
print("Conclusão: não é o calor que gera ocorrência, é a estação chuvosa.")
print("Correlação não é causalidade, demonstrado com os próprios dados.")

---
## 2.9 Síntese: o diagnóstico da cadeia

Os quatro elos foram testados. O painel abaixo resume o estado de cada um.

In [ ]:
marcar(29, "Síntese — diagnóstico dos quatro elos da cadeia de valor")

sintese = pd.DataFrame([
    {"elo": "1. Detecção", "pergunta": "Sensores detectam mais rápido?",
     "teste": "t de Student", "estatística": f"t = {t_stat:.1f}",
     "valor_p": p_valor, "veredito": "FUNCIONA"},
    {"elo": "2. Despacho", "pergunta": "A fila respeita a prioridade?",
     "teste": "Correlação", "estatística": f"r = {r_pearson:+.3f}",
     "valor_p": p_pearson, "veredito": "QUEBRADO"},
    {"elo": "2. Despacho", "pergunta": "O sensor cumpre mais SLA?",
     "teste": "Qui-quadrado", "estatística": f"V = {v_cramer:.3f}",
     "valor_p": p_qui, "veredito": "QUEBRADO"},
    {"elo": "3. Execução", "pergunta": "O serviço é igual em toda a cidade?",
     "teste": "ANOVA", "estatística": f"F = {f_stat:.1f}",
     "valor_p": p_anova, "veredito": "DESIGUAL"},
    {"elo": "4. Percepção", "pergunta": "O cidadão sente o prazo quebrado?",
     "teste": "Correlação", "estatística": f"r = {comparativo.iloc[0]['pearson_r']:+.3f}",
     "valor_p": comparativo.iloc[0]["valor_p"], "veredito": "CONFIRMADO"},
])
sintese["valor_p"] = sintese["valor_p"].apply(lambda v: f"{v:.2e}" if v < 0.001 else f"{v:.3f}")
sintese

In [ ]:
marcar(30, "Síntese — a cadeia de valor do Centro de Operações Urbanas")

fig, eixo = plt.subplots(figsize=(12, 3.6))
eixo.axis("off")

elos = [
    ("1. DETECÇÃO", f"{automatica.mean():.0f} min vs {cidadao.mean():.0f} min", "FUNCIONA", OLIVA),
    ("2. DESPACHO", f"r = {r_pearson:+.3f} (p = {p_pearson:.2f})", "QUEBRADO", TERRACOTA),
    ("3. EXECUÇÃO", f"{amplitude:.0f} p.p. entre regiões", "DESIGUAL", OURO),
    ("4. PERCEPÇÃO", f"r = {comparativo.iloc[0]['pearson_r']:+.3f}", "CONFIRMADO", AZUL),
]

largura, espaco = 0.215, 0.031
for indice, (titulo, metrica, estado, cor) in enumerate(elos):
    x = indice * (largura + espaco)
    eixo.add_patch(plt.Rectangle((x, 0.18), largura, 0.62, facecolor=cor, alpha=0.16,
                                 edgecolor=cor, linewidth=2))
    eixo.text(x + largura / 2, 0.68, titulo, ha="center", fontsize=11,
              fontweight="bold", color=TINTA)
    eixo.text(x + largura / 2, 0.51, metrica, ha="center", fontsize=9.5, color=TINTA)
    eixo.text(x + largura / 2, 0.30, estado, ha="center", fontsize=11,
              fontweight="bold", color=cor)
    if indice < len(elos) - 1:
        eixo.annotate("", xy=(x + largura + espaco - 0.006, 0.49),
                      xytext=(x + largura + 0.006, 0.49),
                      arrowprops={"arrowstyle": "-|>", "color": SUAVE, "linewidth": 2})

eixo.set_xlim(-0.02, 4 * largura + 3 * espaco + 0.02)
eixo.set_ylim(0, 1)
eixo.set_title("Cadeia de valor do COU: o ganho da detecção se perde no despacho",
               fontsize=12, pad=14)

plt.tight_layout()
salvar_figura(30, "sintese-cadeia")
plt.show()

### Conclusão da Parte 2

A investigação percorreu os quatro elos e encontrou um diagnóstico claro.

**O elo 1 funciona.** Sensores detectam em aproximadamente metade do tempo dos canais
humanos, com efeito grande e p praticamente zero.

**O elo 2 está quebrado, e é onde o investimento se perde.** O `score_prioridade` não
tem relação detectável com o tempo de despacho, e a fonte do dado não tem associação
com o cumprimento do SLA. A central atende por ordem de chegada.

**O elo 3 funciona, mas de forma desigual.** A diferença entre a melhor e a pior região
é grande demais para ser variação operacional.

**O elo 4 confirma que o cidadão percebe.** E percebe a promessa quebrada, medida pela
`razao_sla`, mais do que o tempo absoluto.

Três armadilhas foram identificadas e descartadas explicitamente: correlações
tautológicas, falácia ecológica e confundimento sazonal. Nenhuma delas entrou nas
conclusões.

A Parte 3 transforma esse diagnóstico em dashboard e em recomendações para a gestão
pública da cidade Alfa.